# riskcheck on a Colab T4

Runs exactly the same walk-forward evaluation as `riskcheck bench` does on a
laptop CPU, so the two timings are comparable. Set the runtime to **GPU**
(Runtime -> Change runtime type -> T4 GPU) before running.

Two things to watch for, both of which the README reports honestly:

1. **float64 is crippled on a T4.** Consumer NVIDIA cards run double precision
   at 1/32 of single-precision throughput. riskcheck defaults to float64
   because minimum-variance weights need it, so the fair comparison is run
   twice: once in float64 (same numbers as the CPU run) and once in float32
   (faster, but the weights differ).
2. **The default problem is small.** 49 assets and 2,736 rebalance dates is a
   long sequence of tiny matrix operations. That is a latency problem, not a
   throughput problem, and it is the shape of workload a GPU is worst at. The
   last cells scale the problem up to find where that changes.

In [ ]:
!nvidia-smi || echo 'No GPU attached: set Runtime -> Change runtime type -> T4 GPU'

In [ ]:
# Colab ships a CUDA-enabled jax; install riskcheck itself from the repo.
!pip install -q "jax[cuda12]" 2>/dev/null || pip install -q jax
!pip install -q git+https://github.com/daltonoscar0/riskcheck.git
!riskcheck --help

In [ ]:
# Download the French files and build the panels. Takes a couple of minutes,
# almost all of it the 100-portfolio file.
import os

os.makedirs('/content/work', exist_ok=True)
%cd /content/work
!riskcheck fetch
!riskcheck build

In [ ]:
# Hyperparameters must be chosen before anything can be benchmarked, and they
# are chosen on the validation span exactly as on the CPU.
!riskcheck validate

In [ ]:
import json
import subprocess


def bench(asset_set, n_portfolios, x64=True, repeats=2):
    """Run `riskcheck bench` in a fresh process (x64 is set at import time)."""
    env = dict(os.environ, RISKCHECK_X64='1' if x64 else '0')
    out = subprocess.run(
        [
            'riskcheck', 'bench',
            '--asset-set', asset_set,
            '--n-portfolios', str(n_portfolios),
            '--repeats', str(repeats),
            '--out', 'results/bench_gpu.json',
        ],
        env=env, capture_output=True, text=True, check=True,
    )
    record = json.loads(out.stdout)
    print(
        f"{record['backend']:4s} x64={record['float64']!s:5s} "
        f"N={record['n_assets']:3d} P={record['n_portfolios_per_set']:6d}  "
        f"best {record['best_seconds']:7.2f}s  "
        f"(first pass incl. compile {record['first_pass_with_compile_seconds']:.1f}s)"
    )
    return record


records = []

## The headline problem size

This is the configuration the README's CPU timing refers to: 49 industries,
6 estimators, 2,000 portfolios per random set, three horizons.

In [ ]:
records.append(bench('industry49', 2000, x64=True))
records.append(bench('industry49', 2000, x64=False))

## Scaling up

More assets (95 of the 100 size/book-to-market portfolios) and more random
portfolios. If the GPU is ever going to win, it will be here: the quadratic
forms `w' S w` become a genuinely large batched matmul.

In [ ]:
for n_portfolios in (2000, 8000, 20000, 50000):
    for x64 in (True, False):
        records.append(bench('size_bm_100', n_portfolios, x64=x64, repeats=1))

In [ ]:
import pandas as pd

table = pd.DataFrame(records)[
    ['backend', 'float64', 'n_assets', 'n_portfolios_per_set',
     'first_pass_with_compile_seconds', 'best_seconds']
]
print(table.to_markdown(index=False))
table.to_csv('results/bench_gpu_table.csv', index=False)

Paste the resulting table into the README's compute-benchmark section next to
the CPU numbers, including the cases where the GPU loses.